# **Processamento de Dados da API do The Cancer Genome Atlas (TCGA)**
Desenvolvida pela Genomic Data Commons (GDC)

# Importação de Bibliotecas

In [1]:
import json

import pandas as pd
import requests

# Constantes

## API

In [2]:
# URL HTML base da API TCGA / GDC
TCGA_API_ENDPOINT = 'https://api.gdc.cancer.gov'

# Endpoint para verificar status e versão da API
STATUS_ENDPOINT = f'{TCGA_API_ENDPOINT}/status'

# Endpoint de projetos relacionados aos programas
PROJECTS_ENDPOINT = f'{TCGA_API_ENDPOINT}/projects'

# Endpoint de casos relacionados aos projetos
CASES_ENDPOINT = f'{TCGA_API_ENDPOINT}/cases'

# Endpoint de arquivos relacionados aos casos
FILES_ENDPOINT = f'{TCGA_API_ENDPOINT}/files'

## Pasta de Dados e Pré-definições

In [3]:
# Caminho da pasta de dados provisórios
INTERIM_DATA_PATH = '../../data/interim'

# Lista de projetos de interesse
PROJECTS_LIST = [
    'TCGA-KIRP',
    'TCGA-BRCA',
    'TCGA-KICH',
    'TCGA-THCA',
    'TCGA-COAD',
    'TCGA-HNSC',
    'TCGA-KIRC',
    'TCGA-LIHC',
    'TCGA-STAD',
    'TCGA-LUAD',
    'TCGA-UCEC',
    'TCGA-LUSC',
    'TCGA-PRAD',
    'TCGA-ESCA',
    'TCGA-BLCA',
    'TCGA-CHOL',
    'TCGA-PAAD'
]

# Status da API

In [4]:
# Requisita o status e a versão da API ao endpoint
response = requests.get(STATUS_ENDPOINT)

# Imprime o conteúdo da resposta
print(response.content.decode('utf-8'))

{"commit":"3ed4235efe2f2d9c9969e1ce790e6b34a5c60c9d","data_release":"Data Release 41.0 - August 28, 2024","status":"OK","tag":"7.4.0","version":1}



# Projetos

## Obtenção dos Projetos de Interesse

In [5]:
# Campos a serem retornados pela requisição ao endpoint
fields = [
    'disease_type',
    'name',
    'primary_site',
    'summary.case_count'
]
fields = ','.join(fields)

# Filtro usado na requisição ao endpoint
filters = {
    'op': 'in',
    'content': {
        'field': 'project_id',
        'value': PROJECTS_LIST
    }
}

# Parâmetros usados na requisição ao endpoint
params = {
    'fields': fields,
    'filters': filters,
    'size': str(len(PROJECTS_LIST))
}

# Requisita os objetos ao endpoint
response = requests.post(
    url=PROJECTS_ENDPOINT,
    headers={'Content-Type': 'application/json'},
    json=params
)

# Converte o conteúdo da resposta da requisição para JSON
json_response = json.loads(response.content.decode('utf-8'))

## Criação do DataFrame de Projetos

In [6]:
# Cria um DataFrame de projetos de interesse
df_projects = pd.json_normalize(json_response['data']['hits']) \
    .rename(
        columns={
            'id': 'project_id',
            'disease_type': 'project_disease_type',
            'name': 'project_name',
            'primary_site': 'project_primary_site',
            'summary.case_count': 'project_case_count'
        }
    )

In [7]:
# Imprime o DataFrame de projetos de interesse
pd.set_option('display.max_colwidth', 500)
df_projects

,project_id,project_primary_site,project_disease_type,project_name,project_case_count
0,TCGA-PAAD,[Pancreas],"[Cystic, Mucinous and Serous Neoplasms, Ductal and Lobular Neoplasms, Adenomas and Adenocarcinomas, Epithelial Neoplasms, NOS]",Pancreatic Adenocarcinoma,185
1,TCGA-STAD,[Stomach],"[Cystic, Mucinous and Serous Neoplasms, Adenomas and Adenocarcinomas]",Stomach Adenocarcinoma,443
2,TCGA-PRAD,[Prostate gland],"[Cystic, Mucinous and Serous Neoplasms, Adenomas and Adenocarcinomas, Ductal and Lobular Neoplasms]",Prostate Adenocarcinoma,500
3,TCGA-KICH,[Kidney],[Adenomas and Adenocarcinomas],Kidney Chromophobe,113
4,TCGA-HNSC,"[Larynx, Oropharynx, Other and unspecified parts of mouth, Base of tongue, Lip, Tonsil, Floor of mouth, Other and unspecified parts of tongue, Hypopharynx, Gum, Other and ill-defined sites in lip, oral cavity and pharynx, Bones, joints and articular cartilage of other and unspecified sites, Palate]",[Squamous Cell Neoplasms],Head and Neck Squamous Cell Carcinoma,528
5,TCGA-LUAD,[Bronchus and lung],"[Cystic, Mucinous and Serous Neoplasms, Adenomas and Adenocarcinomas, Acinar Cell Neoplasms]",Lung Adenocarcinoma,585
6,TCGA-LIHC,[Liver and intrahepatic bile ducts],[Adenomas and Adenocarcinomas],Liver Hepatocellular Carcinoma,377
7,TCGA-LUSC,[Bronchus and lung],[Squamous Cell Neoplasms],Lung Squamous Cell Carcinoma,504
8,TCGA-KIRP,[Kidney],[Adenomas and Adenocarcinomas],Kidney Renal Papillary Cell Carcinoma,291
9,TCGA-KIRC,[Kidney],[Adenomas and Adenocarcinomas],Kidney Renal Clear Cell Carcinoma,537


# Casos

## Obtenção dos Casos de Interesse

In [8]:
# Campos a serem retornados pela requisição ao endpoint
fields = [
    'disease_type',
    'files.created_datetime',
    'files.data_format',
    'files.data_type',
    'files.file_id',
    'files.experimental_strategy',
    'files.updated_datetime',
    'primary_site',
    'project.project_id'
]
fields = ','.join(fields)

# Filtros usados na requisição ao endpoint
filters = {
    'op': 'and',
    'content': [
        {
            'op': 'in',
            'content': {
                'field': 'project.project_id',
                'value': PROJECTS_LIST
            }
        },
        {
            'op': 'in',
            'content': {
                'field': 'files.experimental_strategy',
                'value': ['miRNA-Seq', 'RNA-Seq']
            }
        }
    ]
}

# Parâmetros usados na requisição ao endpoint
params = {
    'fields': fields,
    'filters': filters,
    'size': str(df_projects['project_case_count'].sum())
}

# Requisita os objetos ao endpoint
response = requests.post(
    url=CASES_ENDPOINT,
    headers={'Content-Type': 'application/json'},
    json=params
)

# Converte o conteúdo da resposta da requisição para JSON
json_response = json.loads(response.content.decode('utf-8'))

## Criação de DataFrames de Casos

In [9]:
# Cria o DataFrame de casos de interesse com arquivos associados
df_cases_and_files = pd.json_normalize(json_response['data']['hits']) \
    .rename(
        columns={
            'disease_type': 'case_disease_type',
            'id': 'case_id',
            'primary_site': 'case_primary_site',
            'project.project_id': 'project_id'
        }
    )

# Cria o DataFrame de casos de interesse
df_cases = df_cases_and_files.drop(columns=['files'])

In [10]:
# Imprime o DataFrame de casos de interesse com arquivos associados
df_cases_and_files

,case_id,case_primary_site,case_disease_type,files,project_id
0,e7e95597-ef88-4442-8c01-8d633822e7b5,Bladder,Transitional Cell Papillomas and Carcinomas,"[{'data_format': 'TSV', 'updated_datetime': '2024-07-30T09:05:09.943588-05:00', 'file_id': '888f59fd-c72b-4d16-b723-1cd27288d496', 'data_type': 'Splice Junction Quantification', 'experimental_strategy': 'RNA-Seq', 'created_datetime': '2021-12-13T18:16:17.014674-06:00'}, {'data_format': 'BCR Biotab', 'updated_datetime': '2018-11-01T15:06:10.843096-05:00', 'file_id': 'dd82f8ca-7e2c-41dd-a7d0-620f07c69b4a', 'data_type': 'Biospecimen Supplement', 'created_datetime': '2018-05-21T12:58:34.430088-0...",TCGA-BLCA
1,06f936e9-5a90-40d3-b91a-713f2b4e6e11,Bladder,Transitional Cell Papillomas and Carcinomas,"[{'data_format': 'BCR Biotab', 'updated_datetime': '2018-11-01T15:06:10.843096-05:00', 'file_id': 'dd82f8ca-7e2c-41dd-a7d0-620f07c69b4a', 'data_type': 'Biospecimen Supplement', 'created_datetime': '2018-05-21T12:58:34.430088-05:00'}, {'data_format': 'BCR Biotab', 'updated_datetime': '2018-11-01T15:06:10.843096-05:00', 'file_id': '2f8175ee-c8fc-49e2-9934-26f78f18c6d5', 'data_type': 'Biospecimen Supplement', 'created_datetime': '2018-05-21T12:58:34.430088-05:00'}, {'data_format': 'TSV', 'updat...",TCGA-BLCA
2,ebedab7c-dd7c-448e-8324-78c86f13be8b,Bladder,Transitional Cell Papillomas and Carcinomas,"[{'data_format': 'BCR Biotab', 'updated_datetime': '2018-11-01T15:06:10.843096-05:00', 'file_id': 'dd82f8ca-7e2c-41dd-a7d0-620f07c69b4a', 'data_type': 'Biospecimen Supplement', 'created_datetime': '2018-05-21T12:58:34.430088-05:00'}, {'data_format': 'TXT', 'updated_datetime': '2023-07-18T07:05:34.549821-05:00', 'file_id': 'ace3794b-50c6-4cfc-a810-a69e6493c6bd', 'data_type': 'Methylation Beta Value', 'experimental_strategy': 'Methylation Array', 'created_datetime': '2023-07-13T06:43:46.437104...",TCGA-BLCA
3,1029514b-a32c-4c43-a440-a5ba2709f717,Bladder,Transitional Cell Papillomas and Carcinomas,"[{'data_format': 'IDAT', 'updated_datetime': '2023-07-18T07:05:34.549821-05:00', 'file_id': '08f8d2d6-c446-4711-8fa7-8a5fcfc81b0c', 'data_type': 'Masked Intensities', 'experimental_strategy': 'Methylation Array', 'created_datetime': '2023-07-13T07:15:02.135309-05:00'}, {'data_format': 'PDF', 'updated_datetime': '2022-11-01T11:47:34.546062-05:00', 'file_id': '981b00cb-f1dc-4411-b10c-09118b2f69a3', 'data_type': 'Pathology Report', 'created_datetime': '2022-10-10T15:36:16.292598-05:00'}, {'data...",TCGA-BLCA
4,f3104468-bf17-434b-8a7e-bc3b253121ea,Bladder,Transitional Cell Papillomas and Carcinomas,"[{'data_format': 'BCR Biotab', 'updated_datetime': '2018-11-01T15:06:10.843096-05:00', 'file_id': 'dd82f8ca-7e2c-41dd-a7d0-620f07c69b4a', 'data_type': 'Biospecimen Supplement', 'created_datetime': '2018-05-21T12:58:34.430088-05:00'}, {'data_format': 'BCR Biotab', 'updated_datetime': '2018-11-01T15:06:10.843096-05:00', 'file_id': '2f8175ee-c8fc-49e2-9934-26f78f18c6d5', 'data_type': 'Biospecimen Supplement', 'created_datetime': '2018-05-21T12:58:34.430088-05:00'}, {'data_format': 'BCR Biotab',...",TCGA-BLCA
...,...,...,...,...,...
7183,a798a8cc-4e72-4a8c-9e20-74a14deafd12,Corpus uteri,Adenomas and Adenocarcinomas,"[{'data_format': 'BCR Biotab', 'updated_datetime': '2018-11-01T15:06:10.843096-05:00', 'file_id': '3eefa637-ec64-42fa-8d0d-c740de106669', 'data_type': 'Biospecimen Supplement', 'created_datetime': '2018-05-22T05:20:16.511251-05:00'}, {'data_format': 'TXT', 'updated_datetime': '2023-07-19T10:41:34.809154-05:00', 'file_id': 'a5840dde-219b-48f4-82e1-152bda7e39eb', 'data_type': 'Methylation Beta Value', 'experimental_strategy': 'Methylation Array', 'created_datetime': '2023-07-10T13:49:10.568835...",TCGA-UCEC
7184,fe2e89f7-8f4d-420a-a551-4877cf0fd1d3,Corpus uteri,Adenomas and Adenocarcinomas,"[{'data_format': 'SVS', 'updated_datetime': '2021-10-14T18:40:28.805704-05:00', 'file_id': '0d29b22e-3f4f-462a-bc73-c93237cb3867', 'data_type': 'Slide Image', 'experimental_strategy': 'Tissue Slide', 'created_datetime'

In [11]:
# Imprime o DataFrame de casos de interesse
df_cases

,case_id,case_primary_site,case_disease_type,project_id
0,e7e95597-ef88-4442-8c01-8d633822e7b5,Bladder,Transitional Cell Papillomas and Carcinomas,TCGA-BLCA
1,06f936e9-5a90-40d3-b91a-713f2b4e6e11,Bladder,Transitional Cell Papillomas and Carcinomas,TCGA-BLCA
2,ebedab7c-dd7c-448e-8324-78c86f13be8b,Bladder,Transitional Cell Papillomas and Carcinomas,TCGA-BLCA
3,1029514b-a32c-4c43-a440-a5ba2709f717,Bladder,Transitional Cell Papillomas and Carcinomas,TCGA-BLCA
4,f3104468-bf17-434b-8a7e-bc3b253121ea,Bladder,Transitional Cell Papillomas and Carcinomas,TCGA-BLCA
...,...,...,...,...
7183,a798a8cc-4e72-4a8c-9e20-74a14deafd12,Corpus uteri,Adenomas and Adenocarcinomas,TCGA-UCEC
7184,fe2e89f7-8f4d-420a-a551-4877cf0fd1d3,Corpus uteri,Adenomas and Adenocarcinomas,TCGA-UCEC
7185,db3a4986-55d5-4ecc-be73-59725dce3c33,Corpus uteri,Adenomas and Adenocarcinomas,TCGA-UCEC
7186,ff6b5fc8-0572-4b58-b3a5-bcda41badbc8,Corpus uteri,"Cystic, Mucinous and Serous Neoplasms",TCGA-UCEC


# Arquivos

## Filtragem dos Arquivos de Interesse

In [12]:
# Explode as listas com dicionário sobre os arquivos dos casos
df_cases_and_files = df_cases_and_files.explode('files')

# Filtra os arquivos relacionados às estratégias experimentais miRNA-Seq e RNA-Seq
key = 'experimental_strategy'
df_cases_and_files = df_cases_and_files[
    df_cases_and_files['files'].apply(
        lambda x: (
            key in x and (x[key] == 'miRNA-Seq' or x[key] == 'RNA-Seq')
        )
    )
]

# Explode o conteúdo dos dicionários sobre os arquivos em colunas
df_cases_and_files = pd.concat(
    objs=[
        df_cases_and_files.reset_index(drop=True),
        pd.json_normalize(df_cases_and_files['files'])
    ],
    axis='columns'
)

# Cria o DataFrame de arquivos de interesse
df_files = df_cases_and_files \
    .query(
        '(data_type == "Gene Expression Quantification") ' +
        'or (data_type == "Isoform Expression Quantification")'
    ) \
    .drop(
        columns=[
            'case_disease_type',
            'case_primary_site',
            'experimental_strategy',
            'files',
            'project_id'
        ]
    ) \
    .reset_index(drop=True)

In [13]:
# Imprime o DataFrame de arquivos de interesse
df_files

,case_id,data_format,updated_datetime,file_id,data_type,created_datetime
0,e7e95597-ef88-4442-8c01-8d633822e7b5,TSV,2024-07-30T09:13:15.356860-05:00,ac728bb8-2b0c-4a0a-9c80-23e0fb768393,Gene Expression Quantification,2021-12-13T18:16:19.205415-06:00
1,e7e95597-ef88-4442-8c01-8d633822e7b5,TXT,2024-07-29T14:17:54.966036-05:00,a6f08f65-910e-446e-b568-fe0159eb4bce,Isoform Expression Quantification,2018-03-20T11:08:47.218083-05:00
2,06f936e9-5a90-40d3-b91a-713f2b4e6e11,TXT,2024-07-29T17:10:54.435236-05:00,d3cd5fe0-d0b7-4ddd-b6f0-5e4ba9d1f199,Isoform Expression Quantification,2018-03-20T11:29:44.359578-05:00
3,06f936e9-5a90-40d3-b91a-713f2b4e6e11,TSV,2024-07-30T09:09:30.048315-05:00,58508378-0fc5-45f9-bf10-1cb171c3e662,Gene Expression Quantification,2021-12-13T18:00:12.510785-06:00
4,ebedab7c-dd7c-448e-8324-78c86f13be8b,TSV,2024-07-30T09:03:56.623052-05:00,1f8056d2-b99e-4a81-bb13-999bda6719d6,Gene Expression Quantification,2021-12-13T18:13:43.935982-06:00
...,...,...,...,...,...,...
15788,db3a4986-55d5-4ecc-be73-59725dce3c33,TXT,2024-07-29T19:14:42.610174-05:00,8f917fde-7f87-4093-80a8-d68622b4e309,Isoform Expression Quantification,2018-03-20T10:51:30.178329-05:00
15789,ff6b5fc8-0572-4b58-b3a5-bcda41badbc8,TXT,2024-07-29T19:06:56.074794-05:00,4b4c5c93-3cd3-43f9-b4d5-acefe969e3c8,Isoform Expression Quantification,2018-03-20T10:10:10.615028-05:00
15790,ff6b5fc8-0572-4b58-b3a5-bcda41badbc8,TSV,2024-07-30T15:58:48.840317-05:00,612e462d-1259-4c06-a2f5-27f4477c7a17,Gene Expression Quantification,2021-12-13T21:26:49.599123-06:00
15791,43476c88-86bc-4a65-9327-f0d24a41c971,TSV,2024-07-30T15:56:46.035507-05:00,36cd1c9b-ab47-4a09-96d6-848cd59c08dd,Gene Expression Quantification,2021-12-13T21:28:20.663448-06:00


## Obtenção dos Dados das Amostras

In [14]:
# Lista com os UUIDs dos arquivos de interesse
files_ids = df_files['file_id'].to_list()

# Campos de interesse para a requisição ao endpoint
fields = [
    'cases.samples.tissue_type',
    'cases.samples.tumor_descriptor',
    'cases.samples.sample_type'
]
fields = ','.join(fields)

# Filtro usado na requisição ao endpoint
filters = {
    'op': 'in',
    'content': {
        'field': 'file_id',
        'value': files_ids
    }
}

# Parâmetros usados na requisição ao endpoint
params = {
    'fields': fields,
    'filters': filters,
    'size': str(len(files_ids))
}

# Requisita o objeto ao endpoint
response = requests.post(
    url=FILES_ENDPOINT,
    headers={'Content-Type': 'application/json'},
    json=params
)

# Converte o conteúdo da resposta da requisição para JSON
json_response = json.loads(response.content.decode('utf-8'))

## Explosão dos Dados de Amostras

In [15]:
# Cria o DataFrame de amostras dos arquivos
df_files_samples = pd.json_normalize(json_response['data']['hits'])

# Explode as listas de dicionários que representam os dados da amostra
df_samples = pd.json_normalize(
    pd.json_normalize(
        pd.json_normalize(
            df_files_samples.explode('cases')['cases']
        )['samples']
    )[0]
)

# Concatena os dados explodidos das amostras aos UUIDs dos arquivos
df_files_samples = pd.concat(
    objs=[df_files_samples, df_samples],
    axis='columns'
)

# Rearranja as colunas do DataFrame
df_files_samples = df_files_samples \
    .rename(columns={'id': 'file_id'}) \
    .drop(columns=['cases'])

In [16]:
# Imprime o DataFrame de amostras dos arquivos
df_files_samples

,file_id,tumor_descriptor,sample_type,tissue_type
0,e1dd7287-ef41-405b-8886-06f27af8649d,Primary,Primary Tumor,Tumor
1,996c69b3-c15a-41bb-8105-f2b4a5b40325,Not Applicable,Solid Tissue Normal,Normal
2,a36c99ec-753c-4859-9082-b077434a3f23,Primary,Primary Tumor,Tumor
3,c7985505-df33-4c0e-98bb-188003ea701f,Primary,Primary Tumor,Tumor
4,590af49d-2e1d-4cad-ab8c-9abe3c7558b0,Primary,Primary Tumor,Tumor
...,...,...,...,...
15788,c3d710c4-88ab-4aae-aa20-8bf5f55fc423,Primary,Primary Tumor,Tumor
15789,e39cedc9-3542-470a-80fe-7d1239e3ff7d,Primary,Primary Tumor,Tumor
15790,a5c60e17-b6e5-4c89-b715-9fe4c6cb3471,Primary,Primary Tumor,Tumor
15791,f9b6ad87-b4a2-46eb-8987-604024b9287f,Primary,Primary Tumor,Tumor


## Enriquecimento do DataFrame de Arquivos

In [17]:
# Enriquece o DataFrame de arquivos de interesse
df_files = df_files \
    .merge(
        right=df_files_samples,
        left_on='file_id',
        right_on='file_id',
        how='inner'
    ) \
    [[
        'file_id',
        'data_format',
        'data_type',
        'sample_type',
        'tissue_type',
        'tumor_descriptor',
        'created_datetime',
        'updated_datetime',
        'case_id'
    ]]

In [18]:
# Imprime o DataFrame enriquecido de arquivos de interesse
df_files

,file_id,data_format,data_type,sample_type,tissue_type,tumor_descriptor,created_datetime,updated_datetime,case_id
0,ac728bb8-2b0c-4a0a-9c80-23e0fb768393,TSV,Gene Expression Quantification,Primary Tumor,Tumor,Primary,2021-12-13T18:16:19.205415-06:00,2024-07-30T09:13:15.356860-05:00,e7e95597-ef88-4442-8c01-8d633822e7b5
1,a6f08f65-910e-446e-b568-fe0159eb4bce,TXT,Isoform Expression Quantification,Primary Tumor,Tumor,Primary,2018-03-20T11:08:47.218083-05:00,2024-07-29T14:17:54.966036-05:00,e7e95597-ef88-4442-8c01-8d633822e7b5
2,d3cd5fe0-d0b7-4ddd-b6f0-5e4ba9d1f199,TXT,Isoform Expression Quantification,Primary Tumor,Tumor,Primary,2018-03-20T11:29:44.359578-05:00,2024-07-29T17:10:54.435236-05:00,06f936e9-5a90-40d3-b91a-713f2b4e6e11
3,58508378-0fc5-45f9-bf10-1cb171c3e662,TSV,Gene Expression Quantification,Primary Tumor,Tumor,Primary,2021-12-13T18:00:12.510785-06:00,2024-07-30T09:09:30.048315-05:00,06f936e9-5a90-40d3-b91a-713f2b4e6e11
4,1f8056d2-b99e-4a81-bb13-999bda6719d6,TSV,Gene Expression Quantification,Primary Tumor,Tumor,Primary,2021-12-13T18:13:43.935982-06:00,2024-07-30T09:03:56.623052-05:00,ebedab7c-dd7c-448e-8324-78c86f13be8b
...,...,...,...,...,...,...,...,...,...
15788,8f917fde-7f87-4093-80a8-d68622b4e309,TXT,Isoform Expression Quantification,Primary Tumor,Tumor,Primary,2018-03-20T10:51:30.178329-05:00,2024-07-29T19:14:42.610174-05:00,db3a4986-55d5-4ecc-be73-59725dce3c33
15789,4b4c5c93-3cd3-43f9-b4d5-acefe969e3c8,TXT,Isoform Expression Quantification,Primary Tumor,Tumor,Primary,2018-03-20T10:10:10.615028-05:00,2024-07-29T19:06:56.074794-05:00,ff6b5fc8-0572-4b58-b3a5-bcda41badbc8
15790,612e462d-1259-4c06-a2f5-27f4477c7a17,TSV,Gene Expression Quantification,Primary Tumor,Tumor,Primary,2021-12-13T21:26:49.599123-06:00,2024-07-30T15:58:48.840317-05:00,ff6b5fc8-0572-4b58-b3a5-bcda41badbc8
15791,36cd1c9b-ab47-4a09-96d6-848cd59c08dd,TSV,Gene Expression Quantification,Primary Tumor,Tumor,Primary,2021-12-13T21:28:20.663448-06:00,2024-07-30T15:56:46.035507-05:00,43476c88-86bc-4a65-9327-f0d24a41c971
